In [ ]:
import seaborn             as sns
import libraries.utilities as sul
import os

from pymatgen.core.surface   import generate_all_slabs
from pymatgen.core.structure import Structure
from pymatgen.io.ase         import AseAtomsAdaptor
from ase.io.vasp             import write_vasp

sns.set_theme()

We only consider coherent interfaces (which has matching lattices from both sides of the interface, meaning that there is a repeat along the interface that lets build periodicity along the interface surface).

In [ ]:
# Define name of folder and path to reference POSCAR
general_folder = 'input/CeO2'
path_to_POSCAR = 'POSCAR-uc-CeO2'

# Defien KPOINTS density
kpoints_density = [40, 40, 40]  # From reference relaxation

# Maximum number for the Miller index in each direction
max_index = 4

# Whether to repair terminations with broken bonds or just omit them
repair = True

# Minimum thicknesses for slab and vacuum
min_slab_size   = 20.0  # Angstroms
min_vacuum_size = 20.0  # Angstroms

The surface formation energy ($E_{surface}$) can be expressed in terms of the area ($S$) of the surface, the energy ($E_{slab}$) of the slab and the energy of the bulk system ($E_{bulk}$) as:

\begin{equation}
    E_{surface} = \frac{E_{slab} - E_{bulk}}{2 S}
\end{equation}

where all energies are per atom.

In [ ]:
# The folder is named as Bi2S3_x_y_z_vi (eg, Bi2S3_0_1_0_v0)
i = 0
while True:
    if not os.path.exists(general_folder):
        # Generate new folder
        os.system(f'mkdir {general_folder}')
    
    slab_folder = f'{general_folder}/slab_v{i}'
    if not os.path.exists(slab_folder):
        # Generate new folder
        os.system(f'mkdir {slab_folder}')

        # Copy POSCAR there (named as unticell, and POSCAR for creating supercell)
        os.system(f'cp {path_to_POSCAR} {slab_folder}/POSCAR')
        break
    i += 1
slab_folder

# ML-IAP relaxation

In [ ]:
# Unit-cell relaxation

# Define paths to bulk and POSCAR
path_to_bulk = f'{slab_folder}/bulk'

# Generate directory for current slab
os.system(f'mkdir {path_to_bulk}')

# Copy POSCAR to new directory
os.system(f'cp {slab_folder}/POSCAR {path_to_bulk}/POSCAR')
os.system(f'cp {slab_folder}/POSCAR {path_to_bulk}/CONTCAR')

# Save KPOINTS with the desired density
sul.generate_kpoints(poscar_file=f'{path_to_bulk}/POSCAR', kpoints_file=f'{path_to_bulk}/KPOINTS', kpoints_density=kpoints_density)

# Read relaxed structure
structure = Structure.from_file(f'{path_to_bulk}/CONTCAR')

# Save slab information
sul.save_json(structure, data='bulk', filename=f'{path_to_bulk}/slab_data.json')

# Slab generation

In [ ]:
# Generate all slabs
slabs = generate_all_slabs(structure,
                           max_index=max_index,
                           min_slab_size=min_slab_size,
                           min_vacuum_size=min_vacuum_size,
                           repair=repair)

In [ ]:
for n, slab in enumerate(slabs):
    print(n, "Polar:", slab.is_polar(), "Symmetric: ", slab.is_symmetric())

Surface formation energy of shape $S$ for slab of energy $E_S$ with $N$ formula units and $E_{bulk}$ bulk energy is:

\begin{equation}
    E_S = \frac{E_S - E_{bulk}}{2 S}
\end{equation}

where all energies are per atom.

In [ ]:
# Initialize the data dictionary for storing all slab energy calculations
# Iterate over slabs
for i, slab in enumerate(slabs):
    print()
    print(f'Slab {i+1}')
    print()
    print(f'Miller index: {slab.miller_index}')
    print(f'Shift: {slab.shift:.3g}')
    print(f'Surface area: {slab.surface_area:.3g}')
    print(f'Number of sites: {len(slab.sites)}')

    # Miller index tuple to string
    miller_index_str = '_'.join(str(element) for element in slab.miller_index)
    miller_index_str = f'{miller_index_str}_i_{i}'

    # Define current folder
    miller_folder = f'{slab_folder}/{miller_index_str}'

    # Generate new folder
    os.system(f'mkdir {miller_folder}')

    # Save slab structure into miller_folder
    write_vasp(f'{miller_folder}/POSCAR', AseAtomsAdaptor.get_atoms(slab), direct=True, sort=True)

    # Save KPOINTS with the desired density
    sul.generate_kpoints(poscar_file=f'{miller_folder}/POSCAR', kpoints_file=f'{miller_folder}/KPOINTS', kpoints_density=kpoints_density)

    # Save slab information
    sul.save_json(slab, filename=f'{miller_folder}/slab_data.json')